# 因子集模型矩阵 v1

目标：把 `src/factor` 的 IC 筛选和组合因子接入 `src/model` 的训练/回测矩阵，比较同一模型在不同因子集上的表现差异。

边界：本轮仍使用 smoke adapter，目的是验证全平台链路：data -> factor -> model -> backtest -> analysis -> notebook sidecar。

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

REPO_ROOT = Path.cwd().resolve()
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / 'src').exists():
        REPO_ROOT = candidate
        break
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.model import run_factor_feature_set_matrix

CARD = REPO_ROOT / 'research/factor_feature_set_matrix_v1'
CARD.mkdir(parents=True, exist_ok=True)

model_names = ('ridge_mse', 'random_forest_clf', 'hist_gradient_mse')
outputs = run_factor_feature_set_matrix(
    card_dir=CARD,
    start='2026-01-01',
    end='2026-06-09',
    train_end='2026-04-30',
    model_names=model_names,
    top_n_factors=5,
)
outputs.summary

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'status': 'feature_set_matrix_completed',
 'base_model_count': 3,
 'feature_set_count': 3,
 'variant_count': 9,
 'start': '2026-01-01',
 'end': '2026-06-09',
 'train_end': '2026-04-30',
 'test_start': '2026-05-01',
 'test_end': '2026-06-08',
 'top_variant': 'random_forest_clf__all_features',
 'adapter_status': 'smoke_adapter',
 'analysis_layers': ['regime_metrics',
  'window_metrics',
  'qlib_risk_metrics',
  'model_effect_deltas'],
 'schema_version': 'factor_feature_set_plan.v1',
 'selection_source': 'train_only_factor_ic',
 'top_n': 5,
 'min_abs_ic': 0.0,
 'factor_count': 14,
 'selected_factors': ['realized_vol_21',
  'vix_change_1d',
  'drawdown_21',
  'soxx_return_5d',
  'soxl_soxx_relative_5d'],
 'composite_column': 'ic_weighted_composite',
 'feature_sets': [{'name': 'all_features',
   'feature_columns': ('soxl_return_1d',
    'soxx_return_1d',
    'soxl_return_5d',
    'soxx_return_5d',
    'soxl_soxx_relative_5d',
    'vix_level',
    'vix_change_1d',
    'intraday_range',
    

In [2]:
outputs.model_comparison.head(15)

,rank,model,base_model,feature_set,return_pct,benchmark_return_pct,relative_return_pct,max_drawdown_pct,sharpe,avg_position,turnover,stability_score
0,1,random_forest_clf__all_features,random_forest_clf,all_features,49.032788,62.147247,-13.114459,21.677243,3.853465,0.740764,3.128315,54.214258
1,2,hist_gradient_mse__top_ic_factors,hist_gradient_mse,top_ic_factors,47.734610,62.147247,-14.412637,24.127776,3.798898,0.714021,2.614679,50.334514
2,3,ridge_mse__all_features,ridge_mse,all_features,44.648453,62.147247,-17.498794,18.743499,4.202316,0.598022,6.458415,50.325407
3,4,ridge_mse__top_ic_factors,ridge_mse,top_ic_factors,46.748915,62.147247,-15.398332,25.409663,3.628952,0.735775,2.489139,46.298079
4,5,random_forest_clf__top_ic_factors,random_forest_clf,top_ic_factors,45.303453,62.147247,-16.843794,26.623644,3.517806,0.744609,1.698610,43.804742
5,6,random_forest_clf__ic_weighted_composite,random_forest_clf,ic_weighted_composite,44.215582,62.147247,-17.931665,26.689677,3.483948,0.741508,1.213005,43.266953
6,7,ridge_mse__ic_weighted_composite,ridge_mse,ic_weighted_composite,42.781382,62.147247,-19.365865,26.761542,3.398451,0.745204,1.646082,40.021799
7,8,hist_gradient_mse__all_features,hist_gradient_mse,all_features,42.932329,62.147247,-19.214918,25.475508,3.493672,0.706046,3.830895,38.362877
8,9,hist_gradient_mse__ic_weighted_composite,hist_gradient_mse,ic_weighted_composite,42.181364,62.147247,-19.965883,27.083518,3.408970,0.698838,3.886951,34.642764


In [3]:
pd.read_csv(CARD / 'feature_set_specs.json') if False else json.loads((CARD / 'feature_set_specs.json').read_text(encoding='utf-8'))

[{'name': 'all_features',
  'feature_columns': ['soxl_return_1d',
   'soxx_return_1d',
   'soxl_return_5d',
   'soxx_return_5d',
   'soxl_soxx_relative_5d',
   'vix_level',
   'vix_change_1d',
   'intraday_range',
   'volume_z_20',
   'trend_5',
   'trend_21',
   'realized_vol_5',
   'realized_vol_21',
   'drawdown_21'],
  'description': '原始 supervised 特征全集。'},
 {'name': 'top_ic_factors',
  'feature_columns': ['realized_vol_21',
   'vix_change_1d',
   'drawdown_21',
   'soxx_return_5d',
   'soxl_soxx_relative_5d'],
  'description': '仅使用训练集 IC 绝对值筛选出的候选因子。'},
 {'name': 'ic_weighted_composite',
  'feature_columns': ['ic_weighted_composite'],
  'description': '使用训练集 IC 作为权重、训练集均值方差标准化后的组合因子。'}]

In [4]:
outputs.model_comparison.groupby('feature_set')[['return_pct', 'relative_return_pct', 'stability_score']].mean().sort_values('stability_score', ascending=False)

,return_pct,relative_return_pct,stability_score
feature_set,,,
all_features,45.537856,-16.609391,47.634181
top_ic_factors,46.595659,-15.551588,46.812445
ic_weighted_composite,43.059442,-19.087804,39.310505


In [5]:
outputs.regime_metrics.head(20)

,model,regime,return_pct,benchmark_return_pct,relative_return_pct,max_drawdown_pct,sharpe,avg_position,turnover,stability_score
0,hist_gradient_mse__all_features,bull,54.649764,94.603219,-39.953455,2.268764,10.006070,0.664075,1.835285,148.203940
1,hist_gradient_mse__all_features,downtrend,59.081503,105.958443,-46.876940,2.655171,18.895469,0.726528,1.352159,242.012909
2,hist_gradient_mse__all_features,neutral,0.000000,0.000000,0.000000,0.000000,0.000000,0.497024,0.000000,0.000000
3,hist_gradient_mse__all_features,stress,30.014909,28.785485,1.229423,7.223241,10.160339,0.840790,0.643451,121.302342
4,hist_gradient_mse__ic_weighted_composite,bull,56.595814,94.603219,-38.007404,2.512408,13.298513,0.795824,1.828400,182.783633
5,hist_gradient_mse__ic_weighted_composite,downtrend,65.375055,105.958443,-40.583388,2.697733,13.481118,0.651720,1.149999,194.514074
6,hist_gradient_mse__ic_weighted_composite,neutral,0.000000,0.000000,0.000000,0.000000,0.000000,0.428443,0.000000,0.000000
7,hist_gradient_mse__ic_weighted_composite,stress,20.138690,39.334432,-19.195742,0.000000,19.250569,0.545467,0.908553,210.827270
8,hist_gradient_mse__top_ic_factors,bull,57.822086,94.603219,-36.781133,2.993307,10.481105,0.701657,1.253219,156.385062
9,hist_gradient_mse__top_ic_factors,downtrend,61.093605,105.958443,-44.864838,3.517424,12.115981,0.669089,0.703973,176.448690


In [6]:
outputs.model_effect_deltas.loc[:, ['segment_type', 'segment', 'model', 'relative_return_pct_delta', 'stability_score_delta', 'diagnosis']].head(20)

,segment_type,segment,model,relative_return_pct_delta,stability_score_delta,diagnosis
0,all,all,random_forest_clf__all_features,0.000000,0.000000,基准模型。
1,all,all,hist_gradient_mse__top_ic_factors,-1.298178,-3.879744,弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。
2,all,all,ridge_mse__top_ic_factors,-2.283873,-7.916179,弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。
3,all,all,random_forest_clf__top_ic_factors,-3.729335,-10.409516,弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。
4,all,all,ridge_mse__all_features,-4.384335,-3.888851,弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。
5,all,all,random_forest_clf__ic_weighted_composite,-4.817205,-10.947305,弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。
6,all,all,hist_gradient_mse__all_features,-6.100459,-15.851381,弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。
7,all,all,ridge_mse__ic_weighted_composite,-6.251406,-14.192459,弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。
8,all,all,hist_gradient_mse__ic_weighted_composite,-6.851424,-19.571494,弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。
9,regime,bull,random_forest_clf__ic_weighted_composite,7.886783,-8.746590,略强于基准：收益有改善，但仍需看稳定性和回撤。


## 中文结论

- 做了什么：执行 3 个代表模型 × 3 套因子集的训练、预测、回测和 `src/analysis` 分层比较。
- 因子集：`all_features`、`top_ic_factors`、`ic_weighted_composite`；筛选来源为 `train_only_factor_ic`，Top N=5。
- 核心指标：最佳 variant 为 `random_forest_clf__all_features`，base model=`random_forest_clf`，feature set=`all_features`，稳定性分数 54.2143。
- regime 差异：`regime_metrics.csv` 和 `model_effect_deltas.csv` 已按 variant 输出压力期/趋势期差异。
- 是否晋升：否。本轮验证因子诊断结果能进入模型矩阵，不代表 smoke adapter 的模型可生产运行。
- 风险：Top IC 因子和组合因子只使用训练集统计，避免测试集泄漏；但单资产时间序列 IC 仍需要后续多资产横截面验证。
- 下一步：把 dashboard 改为读取 feature-set 矩阵 manifest，并增加按 feature set 聚合的模型差异展示。